# CodSoft Task 3 — Iris Flower Classification

This project builds machine learning classification models to classify Iris flowers into their species using sepal and petal measurements.

### Workflow
1. Load and inspect the Iris dataset
2. Clean and validate the data
3. Perform exploratory data analysis
4. Visualize relationships between flower measurements
5. Split the data into training and testing sets
6. Train Logistic Regression
7. Train K-Nearest Neighbors
8. Train Random Forest
9. Compare model performance
10. Evaluate the selected model with a confusion matrix and classification report
11. Make sample predictions

**Target:** `Species`


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


In [ ]:
# Load the dataset
from pathlib import Path

candidate_paths = [
    Path("IRIS.csv"),
    Path("/mnt/data/IRIS.csv")
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "IRIS.csv was not found. Place the CSV in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

# Clean column names and text values
df.columns = [col.strip() for col in df.columns]
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Inspect the dataset
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSpecies distribution:")
display(df["species"].value_counts())


In [ ]:
# Descriptive statistics
display(df.describe(include="all").T)


## Exploratory Data Analysis

In [ ]:
# Species distribution
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="species")
plt.title("Iris Species Distribution")
plt.xlabel("Species")
plt.ylabel("Number of Flowers")
plt.show()


In [ ]:
# Pairplot to visualize relationships between measurements
sns.pairplot(
    df,
    hue="species",
    diag_kind="hist"
)
plt.suptitle("Iris Feature Relationships", y=1.02)
plt.show()


In [ ]:
# Boxplots for each numerical feature
numeric_features = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, feature in zip(axes, numeric_features):
    sns.boxplot(data=df, x="species", y=feature, ax=ax)
    ax.set_title(f"{feature} by Species")
    ax.set_xlabel("Species")
    ax.set_ylabel(feature)

plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix
plt.figure(figsize=(7, 5))
sns.heatmap(
    df[numeric_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Matrix of Iris Measurements")
plt.show()


## Preparing the Data

In [ ]:
X = df[numeric_features]
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


## Model 1 — Logistic Regression

In [ ]:
logistic_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

logistic_model.fit(X_train, y_train)
pred_lr = logistic_model.predict(X_test)

print("Logistic Regression")
print(classification_report(y_test, pred_lr))

lr_accuracy = accuracy_score(y_test, pred_lr)
lr_precision = precision_score(y_test, pred_lr, average="weighted")
lr_recall = recall_score(y_test, pred_lr, average="weighted")
lr_f1 = f1_score(y_test, pred_lr, average="weighted")

print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1 Score :", round(lr_f1, 4))


## Model 2 — K-Nearest Neighbors

In [ ]:
knn_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

knn_model.fit(X_train, y_train)
pred_knn = knn_model.predict(X_test)

print("K-Nearest Neighbors")
print(classification_report(y_test, pred_knn))

knn_accuracy = accuracy_score(y_test, pred_knn)
knn_precision = precision_score(y_test, pred_knn, average="weighted")
knn_recall = recall_score(y_test, pred_knn, average="weighted")
knn_f1 = f1_score(y_test, pred_knn, average="weighted")

print("Accuracy :", round(knn_accuracy, 4))
print("Precision:", round(knn_precision, 4))
print("Recall   :", round(knn_recall, 4))
print("F1 Score :", round(knn_f1, 4))


## Model 3 — Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)

print("Random Forest")
print(classification_report(y_test, pred_rf))

rf_accuracy = accuracy_score(y_test, pred_rf)
rf_precision = precision_score(y_test, pred_rf, average="weighted")
rf_recall = recall_score(y_test, pred_rf, average="weighted")
rf_f1 = f1_score(y_test, pred_rf, average="weighted")

print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))


## Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "K-Nearest Neighbors",
        "Random Forest"
    ],
    "Accuracy": [
        lr_accuracy,
        knn_accuracy,
        rf_accuracy
    ],
    "Precision": [
        lr_precision,
        knn_precision,
        rf_precision
    ],
    "Recall": [
        lr_recall,
        knn_recall,
        rf_recall
    ],
    "F1 Score": [
        lr_f1,
        knn_f1,
        rf_f1
    ]
})

results = results.sort_values("Accuracy", ascending=False).reset_index(drop=True)
display(results.round(4))


In [ ]:
# Select the best model by accuracy
model_predictions = {
    "Logistic Regression": pred_lr,
    "K-Nearest Neighbors": pred_knn,
    "Random Forest": pred_rf
}

best_model_name = results.loc[0, "Model"]
best_predictions = model_predictions[best_model_name]

print("Best model:", best_model_name)


## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, best_predictions, labels=sorted(y.unique()))

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=sorted(y.unique()),
    yticklabels=sorted(y.unique())
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted Species")
plt.ylabel("Actual Species")
plt.show()


## Example Predictions

In [ ]:
# Predict species for a few test samples
sample = X_test.head(10).copy()
sample_predictions = model_predictions[best_model_name][:10]

prediction_output = sample.copy()
prediction_output["Actual_Species"] = y_test.head(10).values
prediction_output["Predicted_Species"] = sample_predictions

display(prediction_output)


## Custom Flower Prediction

You can enter sepal and petal measurements below to predict the Iris species.


In [ ]:
# Example custom flower
new_flower = pd.DataFrame({
    "sepal_length": [5.1],
    "sepal_width": [3.5],
    "petal_length": [1.4],
    "petal_width": [0.2]
})

if best_model_name == "Logistic Regression":
    prediction = logistic_model.predict(new_flower)[0]
elif best_model_name == "K-Nearest Neighbors":
    prediction = knn_model.predict(new_flower)[0]
else:
    prediction = rf_model.predict(new_flower)[0]

print("Predicted species:", prediction)


## Conclusion

The Iris dataset was used to build classification models that identify the species of an Iris flower from its sepal and petal measurements.

Three classification algorithms were trained and compared: Logistic Regression, K-Nearest Neighbors, and Random Forest. The models were evaluated using accuracy, precision, recall, and F1-score.

The model with the highest test accuracy was selected as the final model. A confusion matrix and sample predictions were also generated to demonstrate the model's classification performance.

This completes the core requirements of CodSoft Task 3 — Iris Flower Classification.
